# TorchSig Bounding Box Validation Quality Checks

Two independent checks on the bounding-box labels TorchSig can attach to signals it generates, for **each signal class** in the library.

| | What it does | What you get |
|---|---|---|
| **Task 1: Manual Verification** | One clean scene per class: single signal, no impairments or Component Transforms, very high SNR. Plots the spectrogram with its box overlaid and writes a PNG. | A folder of images to visually inspect. |
| **Task 2: Automatic Verification** | Configurable number of scenes per class. Independently estimates energy fraction within bounding box and the slack in both time and frequency. | Per-class and overall statistics. |

In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import Rectangle

from torchsig.datasets import TorchSigIterableDataset
from torchsig.signals.signal_lists import TorchSigSignalLists
from torchsig.transforms.metadata_transforms import YOLOLabel
from torchsig.transforms.transforms import Spectrogram

### Global Configuration

In [ ]:
SEED = 1234567890  # seed for random generators
IMAGES_DIR = "bbox_images"  # directory to place images

# Scene variables
SAMPLE_RATE = 10e6  # Hz
FFT_SIZE = 512  # FFT dimension
NUM_IQ_SAMPLES = 512 * 512  # 262144 -> a 512x512 spectrogram at fft_size 512
SNR_DB = 50.0  # very high SNR so a signal dominates a scene
SIGNAL_DURATION_MIN = 262144 * 0.8  # samples
SIGNAL_DURATION_MAX = 262144 * 1.0  # samples
SIGNAL_BANDWIDTH_MIN = 2500000  # Hz
SIGNAL_BANDWIDTH_MAX = 3333333  # Hz

# dataset metadata
METADATA = {
    "num_iq_samples_dataset": NUM_IQ_SAMPLES,
    "num_signals_min": 1,
    "num_signals_max": 1,
    "fft_size": FFT_SIZE,
    "fft_stride": FFT_SIZE,
    "sample_rate": SAMPLE_RATE,
    "noise_power_db": 0.0,
    "snr_db_min": SNR_DB,
    "snr_db_max": SNR_DB,
    "cochannel_overlap_probability": 0.0,
    "signal_duration_in_samples_min": SIGNAL_DURATION_MIN,
    "signal_duration_in_samples_max": SIGNAL_DURATION_MAX,
    "bandwidth_min": SIGNAL_BANDWIDTH_MIN,
    "bandwidth_max": SIGNAL_BANDWIDTH_MAX,
    "signal_center_freq_min": -2500000,
    "signal_center_freq_max": 2499999,
    "frequency_min": -2500000,
    "frequency_max": 2499999,
}

# Task 2 variables
NUM_SCENES_PER_CLASS = 100  # spectrograms generated per signal class
TARGET_CONTAINMENT = 0.997  # containment a trimmed box must still hold (3 sigma)
TRIM_WARN = 0.10  # flag a class that could give back this much
TARGET_LO, TARGET_HI = 0.95, 0.99  # band a healthy box should land in


# Task 2 outlier capture
OUTLIER_DIR = "bbox_images_outliers"
MAX_OUTLIERS_PER_CLASS = 10  # cap on images written per class (all are counted)
OUTLIER_REFERENCE_BOX = True  # also draw where the energy actually is

# location to write generated images
os.makedirs(IMAGES_DIR, exist_ok=True)
os.makedirs(OUTLIER_DIR, exist_ok=True)

### Support Functions

In [ ]:
def to_boxes(label):
    """Normalize a yolo_label target into an (N, 5) array of (cid, cx, cy, w, h)."""
    if label is None:
        return np.empty((0, 5))
    arr = np.asarray(label, dtype=float)
    if arr.ndim == 1:
        # single box (5,) -> (1,5); empty/degenerate -> (0,5)
        arr = arr.reshape(1, 5) if arr.size == 5 else arr.reshape(0, 5)
    return arr


def yolo_to_pixel_box(cx, cy, w, h, W, H):
    """Convert a normalized YOLO box to matplotlib imshow data coordinates.

    Time:  frames tile [0, num_iq) with no overlap, so frame j covers samples
           [j*N, (j+1)*N) and is centered at normalized time (j+0.5)/W.
           Inverting, normalized t -> column j = t*W - 0.5.

    Freq:  after fftshift + [::-1], row i holds frequency (H/2 - 1 - i)*fs/H,
           so row i = H/2 - 1 - f*H/fs. YOLOLabel emits y = 0.5 - f/fs,
           which assumes a symmetric [-fs/2, +fs/2] axis. Substituting gives
           row = y*H - 1.0.

    Returns ((x_lower_left, y_upper_left), width_px, height_px) for Rectangle.
    """
    x_center = cx * W - 0.5
    y_center = cy * H - 1.0
    box_w = w * W
    box_h = h * H
    return (x_center - box_w / 2.0, y_center - box_h / 2.0), box_w, box_h


def energy_ratio(spec_db, box):
    """Fraction of a scene's total energy that falls inside its bounding box.

    One signal per scene at very high SNR, so the noise floor carries well under
    0.01 % of the energy and the whole spectrogram is effectively that one
    signal. No noise subtraction, which also keeps the ratio inside [0, 1].
    """
    power = 10.0 ** (np.asarray(spec_db, dtype=np.float64) / 10.0)
    H, W = power.shape

    cid, cx, cy, w, h = box
    (x0, y0), box_w, box_h = yolo_to_pixel_box(cx, cy, w, h, W, H)
    c0, c1 = max(int(round(x0)), 0), min(int(round(x0 + box_w)), W)
    r0, r1 = max(int(round(y0)), 0), min(int(round(y0 + box_h)), H)

    return power[r0:r1, c0:c1].sum() / power.sum()


def _axis_slack(profile, lo, hi, target):
    """Containment of the span [lo, hi) in a 1-D energy profile, and how far
    that span could shrink and still hold `target` of the profile's energy.

    Returns (containment, narrowest_width_px, fractional_reduction).
    """
    if hi <= lo:
        return 0.0, 0, 0.0

    cum = np.concatenate(([0.0], np.cumsum(profile)))
    cum /= cum[-1]

    contained = cum[hi] - cum[lo]
    if contained < target:  # nothing to trim
        return contained, hi - lo, 0.0

    a = np.arange(lo, hi)  # every candidate left edge
    b = np.searchsorted(cum, cum[a] + target)  # tightest matching right edge
    keep = b <= hi  # trimmed span stays in the box
    narrowest = int((b[keep] - a[keep]).min())
    return contained, narrowest, 1.0 - narrowest / (hi - lo)


def box_slack(spec_db, box, target=TARGET_CONTAINMENT):
    """Per-axis containment of a box, and the width/height reduction that would
    still leave `target` of the energy on that axis.

    Time uses the column energy profile (summed over all rows), frequency the
    row profile (summed over all columns), so the axes are measured separately.
    """
    power = 10.0 ** (np.asarray(spec_db, dtype=np.float64) / 10.0)
    H, W = power.shape

    cid, cx, cy, w, h = box
    (x0, y0), box_w, box_h = yolo_to_pixel_box(cx, cy, w, h, W, H)
    c0, c1 = max(int(round(x0)), 0), min(int(round(x0 + box_w)), W)
    r0, r1 = max(int(round(y0)), 0), min(int(round(y0 + box_h)), H)

    t_cont, t_px, t_red = _axis_slack(power.sum(axis=0), c0, c1, target)
    f_cont, f_px, f_red = _axis_slack(power.sum(axis=1), r0, r1, target)

    return {"time_containment": t_cont, "time_reduction": t_red, "time_px": t_px, "freq_containment": f_cont, "freq_reduction": f_red, "freq_px": f_px}


def is_outlier(ratio, slack, threshold=None):
    """True when the energy ratio or either axis containment falls below the band.

    Only the *containment* entries of box_slack are compared. The *reduction*
    entries are a different quantity (how much the box could shrink), and would
    sit below TARGET_LO on essentially every healthy scene.
    """
    threshold = TARGET_LO if threshold is None else threshold
    return min(ratio, slack["time_containment"], slack["freq_containment"]) < threshold


def tight_span(profile, target):
    """Narrowest [lo, hi) span of a 1-D energy profile holding `target` of it.

    Same search as _axis_slack but over the whole axis instead of inside the
    label, and it returns the position rather than just the width, so the span
    can be drawn next to the label for comparison.
    """
    cum = np.concatenate(([0.0], np.cumsum(profile)))
    cum /= cum[-1]
    a = np.arange(profile.size)
    b = np.searchsorted(cum, cum[a] + target)
    keep = b <= profile.size
    a, b = a[keep], b[keep]
    width = b - a
    tie = width == width.min()
    a, b = a[tie], b[tie]
    k = int((cum[b] - cum[a]).argmax())
    return int(a[k]), int(b[k])


def save_outlier_png(spec_db, box, cls, scene_idx, ratio, slack, noise_power_db, outdir=OUTLIER_DIR, reference=OUTLIER_REFERENCE_BOX):
    """Write one annotated spectrogram for a scene that failed the check.

    Solid red box is the label under test. Dashed cyan box, when enabled, is the
    tightest span that actually holds TARGET_CONTAINMENT of the energy on each
    axis, searched over the whole spectrogram rather than inside the label.
    Comparing the two is what separates the two explanations:

      * cyan much larger than red -> the signal really does spill outside the
        label, so the generator or the bandwidth/duration field is at fault
      * cyan similar in size but offset -> the label is misplaced, or the
        pixel mapping in yolo_to_pixel_box is off
      * cyan and red nearly identical -> the label is fine and the low number
        is a measurement artifact, e.g. a box only a few pixels tall being
        rounded to integer rows, or a box running off the image edge
    """
    power = 10.0 ** (np.asarray(spec_db, dtype=np.float64) / 10.0)
    H, W = power.shape

    cid, cx, cy, w, h = box
    (x0, y0), box_w, box_h = yolo_to_pixel_box(cx, cy, w, h, W, H)
    c0, c1 = max(int(round(x0)), 0), min(int(round(x0 + box_w)), W)
    r0, r1 = max(int(round(y0)), 0), min(int(round(y0 + box_h)), H)

    note = (
        f"energy ratio     {100 * ratio:7.2f}%\n"
        f"time containment {100 * slack['time_containment']:7.2f}%\n"
        f"freq containment {100 * slack['freq_containment']:7.2f}%\n"
        f"time reduction   {100 * slack['time_reduction']:7.1f}%\n"
        f"freq reduction   {100 * slack['freq_reduction']:7.1f}%\n"
        f"label box        {box_w:.0f} x {box_h:.0f} px"
        f"  at ({x0:.0f}, {y0:.0f})"
    )

    if (x0 < -0.5) or (x0 + box_w > W - 0.5) or (y0 < -0.5) or (y0 + box_h > H - 0.5):
        note += "\nNOTE: label runs off the image edge"

    ref = None
    if reference:
        rc0, rc1 = tight_span(power.sum(axis=0), TARGET_CONTAINMENT)
        rr0, rr1 = tight_span(power.sum(axis=1), TARGET_CONTAINMENT)
        ref = (rc0, rc1, rr0, rr1)
        note += f"\nactual span (cyan) {rc1 - rc0} x {rr1 - rr0} px  at ({rc0}, {rr0})  holds {100 * TARGET_CONTAINMENT:.1f}%"

    fig, ax = plt.subplots(figsize=(7.5, 6))
    im = ax.imshow(spec_db, aspect="auto", origin="upper", cmap="viridis", vmin=noise_power_db, vmax=noise_power_db + SNR_DB)

    # Identical to the Task 1 rectangle: same yolo_to_pixel_box, same Rectangle
    # arguments, same imshow extent, so a box drawn here lands on exactly the
    # same pixels it would in bbox_images/.
    ax.add_patch(Rectangle((x0, y0), box_w, box_h, fill=False, edgecolor="red", lw=1.2, clip_on=True))

    if ref is not None:
        rc0, rc1, rr0, rr1 = ref
        # integer span [lo, hi) covers pixels lo..hi-1, i.e. data [lo-0.5, hi-0.5)
        ax.add_patch(Rectangle((rc0 - 0.5, rr0 - 0.5), rc1 - rc0, rr1 - rr0, fill=False, edgecolor="cyan", lw=1.0, ls="--", clip_on=True))

    legend = "red = label" + (", cyan = actual" if ref is not None else "")
    ax.set_title(f"{cls}   scene {scene_idx}   energy {100 * ratio:.2f}%   ({legend})", fontsize=10)
    ax.set_xlabel("Time")
    ax.set_ylabel("Frequency")
    cbar = fig.colorbar(im, ax=ax, pad=0.02, extend="both")
    cbar.set_label("Power (dB)")

    # annotation goes below the axes: inside the axes it can cover the very
    # energy spill the image exists to show
    ax.text(0.0, -0.16, note, transform=ax.transAxes, va="top", ha="left", fontsize=8, family="monospace", bbox=dict(facecolor="0.95", edgecolor="0.7", pad=5))

    fig.tight_layout()
    path = f"{outdir}/{cls}_scene{scene_idx:04d}_r{100 * ratio:05.1f}.png"
    fig.savefig(path, dpi=120, bbox_inches="tight")  # include the note
    plt.close(fig)
    return path

### Task 1: Manual Verification

In [ ]:
# Task 1 Baseline: no impairments or Component Transforms

# for each Signal of Interest, create a dataset, then generate an image
for i, cls in enumerate(TorchSigSignalLists.all_signals):
    ds = TorchSigIterableDataset(
        metadata=METADATA,
        component_transforms=[],
        transforms=[Spectrogram(fft_size=FFT_SIZE), YOLOLabel()],
        target_labels=["yolo_label"],  # yolo label format
        signal_generators=[str(cls)],  # single signal of interest
        validate_init=True,
    )
    ds.seed(SEED + i)  # fresh seed for each dataset/SoI

    # spec_db: wideband spectrogram in power dB
    # label: single element list of (cid,cx,cy,w,h)
    spec_db, label = next(ds)
    H, W = spec_db.shape

    # generate figure
    fig, ax = plt.subplots(figsize=(6, 6))
    im = ax.imshow(spec_db, aspect="auto", origin="upper", cmap="viridis", vmin=ds.noise_power_db, vmax=ds.noise_power_db + SNR_DB)
    ax.set_title(cls)
    ax.set_ylabel("Frequency")
    ax.set_xlabel("Time")
    cbar = fig.colorbar(im, ax=ax, pad=0.02, extend="both")
    cbar.set_label("Power (dB)")

    # add bounding boxes
    for cid, cx, cy, w, h in to_boxes(label):
        (x0, y0), box_w, box_h = yolo_to_pixel_box(cx, cy, w, h, W, H)
        ax.add_patch(Rectangle((x0, y0), box_w, box_h, fill=False, edgecolor="red", lw=1.0, clip_on=True))

    fig.savefig(f"bbox_images/{cls}.png", dpi=120)
    plt.close(fig)

### Task 2: Automatic Verification

In [ ]:
stats = {}  # statistics per SoI class
outliers = []  # one record per scene that failed the check
SLACK_KEYS = ("time_containment", "freq_containment", "time_reduction", "freq_reduction")

for i, cls in enumerate(TorchSigSignalLists.all_signals):
    ds = TorchSigIterableDataset(
        metadata=METADATA,  # same scene config as Task 1
        component_transforms=[],
        transforms=[Spectrogram(fft_size=FFT_SIZE), YOLOLabel()],
        target_labels=["yolo_label"],
        signal_generators=[str(cls)],
        validate_init=True,
    )
    ds.seed(SEED + i)

    ratios, slacks = [], []
    n_out, n_saved = 0, 0
    for j in range(NUM_SCENES_PER_CLASS):
        spec_db, label = next(ds)
        for box in to_boxes(label):
            r = energy_ratio(spec_db, box)
            sl = box_slack(spec_db, box)
            ratios.append(r)
            slacks.append(sl)

            if is_outlier(r, sl):
                n_out += 1
                outliers.append(
                    {
                        "signal": str(cls),
                        "scene": j,
                        "ratio": r,
                        "time_containment": sl["time_containment"],
                        "freq_containment": sl["freq_containment"],
                        "saved": n_saved < MAX_OUTLIERS_PER_CLASS,
                    }
                )
                if n_saved < MAX_OUTLIERS_PER_CLASS:
                    save_outlier_png(spec_db, box, str(cls), j, r, sl, ds.noise_power_db)
                    n_saved += 1
    ratios = np.array(ratios)

    s = {
        "n": ratios.size,
        "mean": ratios.mean(),
        "std": ratios.std(),
        "median": np.median(ratios),
        "min": ratios.min(),
        "max": ratios.max(),
        "ratios": ratios,
    }
    for k in SLACK_KEYS:
        s[k] = np.array([x[k] for x in slacks])
    stats[str(cls)] = s

    print(
        f"{cls!s:<22} energy {100 * s['mean']:6.2f}%    trim required:  "
        f"time {100 * np.median(s['time_reduction']):5.1f}%   "
        f"freq {100 * np.median(s['freq_reduction']):5.1f}%"
        f"   outliers {n_out:3d}" + (f" ({n_saved} saved)" if n_out > n_saved else ""),
        flush=True,
    )

In [ ]:
# statistics
pooled = np.concatenate([s["ratios"] for s in stats.values()])
med = lambda name, key: float(np.median(stats[name][key]))


def status_of(name):
    s = stats[name]
    flags = []
    if float(np.median(s["ratios"])) < TARGET_LO:
        flags.append("SMALL")
    if med(name, "time_reduction") > TRIM_WARN:
        flags.append("WIDE")
    if med(name, "freq_reduction") > TRIM_WARN:
        flags.append("TALL")
    return " ".join(flags) if flags else "ok"


hdr = f"{'signal':<22}{'n':>4}{'energy':>9}{'time':>9}{'freq':>9}{'t-trim':>9}{'f-trim':>9}   status"
print(hdr)
print("-" * len(hdr))
print("(energy/time/freq = median containment;  t-trim/f-trim = median width or")
print(f" height that could be given back and still hold {100 * TARGET_CONTAINMENT:.0f}% on that axis)\n")

for name in sorted(stats, key=lambda n: np.median(stats[n]["ratios"])):
    s = stats[name]
    print(
        f"{name:<22}{s['n']:>4}"
        f"{100 * np.median(s['ratios']):>8.2f}%"
        f"{100 * med(name, 'time_containment'):>8.2f}%"
        f"{100 * med(name, 'freq_containment'):>8.2f}%"
        f"{100 * med(name, 'time_reduction'):>8.1f}%"
        f"{100 * med(name, 'freq_reduction'):>8.1f}%   {status_of(name)}"
    )

t_red = np.concatenate([s["time_reduction"] for s in stats.values()])
f_red = np.concatenate([s["freq_reduction"] for s in stats.values()])
in_band = (pooled >= TARGET_LO) & (pooled <= TARGET_HI)
ok_classes = sum(status_of(n) == "ok" for n in stats)

print(f"\n{pooled.size} scenes over {len(stats)} classes")
print(f"energy ratio, pooled mean / median : {100 * pooled.mean():.2f}% / {100 * np.median(pooled):.2f}%")
print(f"in {100 * TARGET_LO:.0f}-{100 * TARGET_HI:.0f}% band                    : {100 * in_band.mean():.1f}% of scenes")
print(f"trim required, median time / freq  : {100 * np.median(t_red):.1f}% / {100 * np.median(f_red):.1f}%")
print(f"trim required, worst time / freq   : {100 * t_red.max():.1f}% / {100 * f_red.max():.1f}%")
print(f"classes clean on both checks       : {ok_classes}/{len(stats)}")

In [ ]:
# Task 2 outliers: scenes whose energy ratio or axis containment fell below
# TARGET_LO. Images are in OUTLIER_DIR, worst ratio first in the listing below.
n_scenes = sum(s["n"] for s in stats.values())

if not outliers:
    print(f"no scenes below {100 * TARGET_LO:.0f}% out of {n_scenes}")
else:
    by_class = {}
    for o in outliers:
        by_class.setdefault(o["signal"], []).append(o)

    print(f"{len(outliers)} of {n_scenes} scenes ({100 * len(outliers) / n_scenes:.2f}%) below {100 * TARGET_LO:.0f}%, across {len(by_class)} of {len(stats)} classes")
    print(f"images written to {OUTLIER_DIR}/ (up to {MAX_OUTLIERS_PER_CLASS} per class)\n")

    hdr = f"{'signal':<22}{'count':>6}{'worst':>9}{'median':>9}{'worst time':>12}{'worst freq':>12}"
    print(hdr)
    print("-" * len(hdr))
    for name in sorted(by_class, key=lambda n: min(o["ratio"] for o in by_class[n])):
        g = by_class[name]
        print(
            f"{name:<22}{len(g):>6}"
            f"{100 * min(o['ratio'] for o in g):>8.2f}%"
            f"{100 * np.median([o['ratio'] for o in g]):>8.2f}%"
            f"{100 * min(o['time_containment'] for o in g):>11.2f}%"
            f"{100 * min(o['freq_containment'] for o in g):>11.2f}%"
        )

    print(f"\nworst {min(15, len(outliers))} individual scenes:")
    for o in sorted(outliers, key=lambda o: o["ratio"])[:15]:
        mark = "" if o["saved"] else "   (over the per-class image cap)"
        print(f"  {o['signal']:<22} scene {o['scene']:>4}   energy {100 * o['ratio']:6.2f}%   time {100 * o['time_containment']:6.2f}%   freq {100 * o['freq_containment']:6.2f}%{mark}")

    # which axis is responsible, pooled over every outlier
    t_bad = sum(o["time_containment"] < TARGET_LO for o in outliers)
    f_bad = sum(o["freq_containment"] < TARGET_LO for o in outliers)
    r_bad = sum(o["ratio"] < TARGET_LO for o in outliers)
    print(f"\ntriggered by energy ratio : {r_bad}")
    print(f"triggered by time axis    : {t_bad}")
    print(f"triggered by frequency    : {f_bad}")

In [ ]:
# containment: energy inside bounding box
names = sorted(stats, key=lambda n: np.median(stats[n]["ratios"]))
y = np.arange(len(names))
m = np.array([np.median(stats[n]["ratios"]) for n in names])
err = np.vstack([m - np.array([stats[n]["min"] for n in names]), np.array([stats[n]["max"] for n in names]) - m])

fig, ax = plt.subplots(figsize=(8, max(4.0, 0.20 * len(names) + 1.0)))
ax.axvspan(TARGET_LO, TARGET_HI, color="green", alpha=0.12, label=f"{100 * TARGET_LO:.0f}-{100 * TARGET_HI:.0f}% target")
ax.errorbar(m, y, xerr=err, fmt="o", ms=4, lw=1, capsize=2, color="k")
ax.set_yticks(y)
ax.set_yticklabels(names, fontsize=7)
ax.set_ylim(-1, len(names))
ax.set_xlim(min(0.90, m.min() - 0.03), 1.005)
ax.set_xlabel("energy inside box (median, whiskers min to max)")
ax.set_title("Is the box big enough?")
ax.grid(axis="x", alpha=0.3)
ax.legend(loc="lower right", fontsize=8)
fig.tight_layout()
fig.savefig("bbox_images/containment_by_class.png", dpi=120)
plt.show()

# slack estimates: bounding box slack in time and frequency
names = sorted(stats, key=lambda n: max(med(n, "time_reduction"), med(n, "freq_reduction")))
y = np.arange(len(names))
tr = np.array([med(n, "time_reduction") for n in names])
fr = np.array([med(n, "freq_reduction") for n in names])

fig, ax = plt.subplots(figsize=(8, max(4.0, 0.20 * len(names) + 1.0)))
ax.axvline(100 * TRIM_WARN, color="crimson", lw=1, ls="--", label=f"{100 * TRIM_WARN:.0f}% flag threshold")
ax.scatter(100 * tr, y, s=22, marker="o", facecolors="none", edgecolors="tab:blue", label="time (width)")
ax.scatter(100 * fr, y, s=22, marker="s", facecolors="none", edgecolors="tab:orange", label="frequency (height)")
ax.set_yticks(y)
ax.set_yticklabels(names, fontsize=7)
ax.set_ylim(-1, len(names))
ax.set_xlim(-1.0, 100 * max(0.15, tr.max(), fr.max()) + 3.0)
ax.set_xlabel(f"reduction (%) that would still hold {100 * TARGET_CONTAINMENT:.1f}% on that axis")
ax.set_title("Is the box too big?")
ax.grid(axis="x", alpha=0.3)
ax.legend(loc="lower right", fontsize=8)
fig.tight_layout()
fig.savefig("bbox_images/trim_by_class.png", dpi=120)
plt.show()